# Unsway — Phase 6C integrity-gated extraction

This notebook extracts **train/validation activations from Phase 6 v2 only**. The retired v2 test remains unused; the disjoint Phase 6C initial-only result supplies the eligibility gate. Pressure/control prompts from the replacement holdout stay unopened.

## Update the repository and install dependencies

In [ ]:
from pathlib import Path

repo = Path("/content/Unsway")
if (repo / ".git").is_dir():
    !git -C /content/Unsway pull --ff-only
else:
    !git clone https://github.com/idris404/Unsway.git /content/Unsway
%cd /content/Unsway
!pip install -q -e '.[dev]'

## Verify the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
print(torch.cuda.get_device_name(0))

## Rebuild the frozen datasets

In [ ]:
!python -m unsway.cli.phase1 --config configs/phase1.yaml
!python -m unsway.cli.phase6 --config configs/phase6.yaml --stage data
!python -m unsway.cli.phase6 --config configs/phase6c.yaml --stage data

## Restore or reproduce the Phase 6C gate

Reuse the initial-only gate when it is present in the runtime; otherwise reproduce it deterministically. Pressure/control prompts remain unopened.

In [ ]:
import json
import subprocess
import zipfile

from google.colab import files

gate_report = Path("reports/phase6c_baseline.json")
gate_predictions = Path("data/processed/phase6c_test_initial_predictions.jsonl")
if not (gate_report.is_file() and gate_predictions.is_file()):
    result = subprocess.run(
        [
            "python",
            "-m",
            "unsway.cli.phase6",
            "--config",
            "configs/phase6c.yaml",
            "--stage",
            "baseline",
        ]
    )
    assert result.returncode == 0, result.returncode
gate = json.loads(gate_report.read_text())
assert gate["status"] == "ready_for_frozen_test"
assert gate["test_initial_only"]["metrics"]["overall"]["initial_correct_trials"] == 537
assert gate["test_initial_only"]["pressure_scored"] is False
assert gate["test_initial_only"]["control_scored"] is False
print("Phase 6C initial-only gate is ready.")

## Reproduce the retired v2 train/validation behavior

In [ ]:
import json
import subprocess

v2_report = Path("reports/phase6_baseline.json")
v2_predictions = Path("data/processed/phase6_train_validation_predictions.jsonl")
if not (v2_report.is_file() and v2_predictions.is_file()):
    result = subprocess.run(
        [
            "python",
            "-m",
            "unsway.cli.phase6",
            "--config",
            "configs/phase6.yaml",
            "--stage",
            "baseline",
        ]
    )
    assert result.returncode == 1, result.returncode
retired = json.loads(v2_report.read_text())
assert retired["status"] == "insufficient_test_eligibility"
assert retired["test_initial_only"]["metrics"]["overall"]["initial_correct_trials"] == 299
print("Retired v2 behavior restored; its pressure/control test prompts remain unopened.")

## Extract train/validation activations through the replacement gate

In [ ]:
!python -m unsway.cli.phase6 --config configs/phase6.yaml --stage extract \
    --eligibility-config configs/phase6c.yaml

## Verify and download the extraction bundle

In [ ]:
report = json.loads(Path("reports/phase6_extraction.json").read_text())
assert report["status"] == "train_validation_multilayer_extracted"
assert report["test_examples_extracted"] == 0
assert report["eligibility_gate"]["mode"] == "external_replacement_holdout"
assert report["eligibility_gate"]["initial_correct_trials"] == 537
print(
    json.dumps(
        {
            "shape": report["shape"],
            "split_counts": report["split_counts"],
            "behavior_counts": report["behavior_counts"],
            "eligibility_gate": report["eligibility_gate"],
        },
        indent=2,
    )
)

bundle = Path("/content/phase6c_extraction_bundle.zip")
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for source in [
        Path("data/processed/phase6/multilayer_final_activations.safetensors"),
        Path("data/processed/phase6/multilayer_examples.json"),
        Path("reports/phase6_extraction.json"),
    ]:
        archive.write(source, arcname=source.name)
files.download(str(bundle))